In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# PT-W1 Day 5：Ontology 能补充什么

> **日期**：2026-07-31（周四）
> **周主题**：DDD → Semantic Model
> **今日核心问题**：用 Ontology 视角审视你的已有资产，缺了什么？

---


## 今日对照的真实材料

1. **business-ontology.yaml** — 你已有的模块→能力→场景结构
2. **ADR-006** — 四类对象关系与生命周期影响模型
3. **effect-registry.yaml** — 5 类冻结的 Lifecycle Transition Effect

---


## 一、你已经有什么（比你以为的多）

打开 `business-ontology.yaml`，你会看到这样的结构：

```yaml
资源管理:                    # ← Ontology Module
  sub_functions:
    铺位管理:                # ← Capability Area
      capabilities:
      - asset-resource-rent-control-ui
      scenarios:             # ← Business Scenario
      - name: 铺位建档
      - name: 铺位拆分合并
      - name: 铺位状态管理
      terms:                 # ← Business Vocabulary
      - 铺位
      - 商铺
      - 计租面积
      - 空置
      - 锁定


In [ ]:
这已经不是"一张表"了。你已经有了 **11 个模块、~50 个子功能、上百个场景、几百个业务术语**。

**用 Ontology 视角翻译：**

| business-ontology.yaml 里的 | Ontology 语言 |
|---|---|
| 模块（资源管理/招商管理/合同管理…） | **Domain Module**（业务域） |
| 子功能（铺位管理/品牌管理…） | **Capability Area**（能力区域） |
| capabilities 列表 | **Capability**（能力声明） |
| scenarios | **Business Scenario**（业务场景） |
| terms | **Business Vocabulary**（业务术语表） |

**结论：你的 business-ontology.yaml 就是一个初级 Ontology。** 它已经有了模块→能力→场景的骨架。

---

## 二、Ontology 六维度 vs 你的资产：缺什么

现在用 Ontology 六个维度逐一检查：

### ✅ 维度 1：Entity（世界里有什么东西？）

**你有的**：business-ontology.yaml 的 terms 列表 + MI Domain Model 的 Object Ownership Matrix。

**铺位、楼宇、楼层、合同、商户、品牌、账单、保证金、工单、广告位……** 几十个业务实体已经识别。

**缺什么**：Entity 的**业务定义**。你的 terms 是术语列表，不是 Entity 定义。"铺位"这个词后面没有写：它是什么？它的业务边界是什么？它和"单元""广告位"的根本区别是什么？

> ❌ AI 看到 `shop` 表知道是一条记录
> ❌ AI 看到 terms 列里有"铺位"知道是一个业务词
> ✅ AI 需要："铺位 = 可独立租赁的最小商业空间单元，有唯一编码和计租面积"

### ✅ 维度 2：Identity（它如何被唯一识别？）

**你有的**：表 ID（`shop.id`）+ 编码规则（铺位编码 = 楼宇编码 + 楼层编码 + 流水号）。

**缺什么**：**业务身份层级**。你的编码规则是技术编码，但业务身份是什么？


技术身份：shop.id = 12345（数据库主键）
编码身份：B1-F2-015（铺位编码）
业务身份：???


In [ ]:
Ontology 视角下的业务身份应该是：


项目（万象城）
  └── 楼宇（A 栋）
       └── 楼层（L2）
            └── 铺位（L2-015）← 这才是业务身份


In [ ]:
**为什么 AI 需要这个？** 当用户问"A 栋二楼那个奶茶店旁边的是哪个铺位"，AI 需要通过层级关系定位到具体 Entity，而不是查数据库 ID。

### ✅ 维度 3：Relationship（事物如何连接？）

**你有的**：ADR-006 定义了四类对象关系——Identity Reference / Structural Composition / Hierarchical Containment / Lifecycle Transition Effect。

**这是一个很强的资产。** 大部分 ERP 系统连关系分类都没做过。

**缺什么**：关系的**语义命名**。ADR-006 定义了关系的**类别**（四类），但没给每对关系一个**动词**：

| 现在的表达 | Ontology 需要的表达 |
|---|---|
| 合同.shop_id = FK 引用 | Lease **occupies** Space |
| 合同.tenant_id = FK 引用 | Lease **is-signed-by** Tenant |
| 铺位.building_id = FK 引用 | Space **is-located-in** Building |
| 合同生效 → effect | Lease.activated **releases-effect** → Space: occupied |

> **FK（外键）告诉 AI "两个表有连接"，但不告诉 AI "它们是什么关系"。**
> 语义关系名（occupies / signs / governs）才是 AI 推理的基础。

### ✅ 维度 4：State / Event（业务如何变化？）

**你有的**：effect-registry.yaml 已经冻结了 5 类 Effect：

| effect_type | 语义 | 注册域 |
|---|---|---|
| state-transition-effect | 对象状态传播 | 通用 |
| occupancy-effect | 资源占用变化 | 租赁 |
| financial-effect | 财务影响 | 财务 |
| lead-conversion-effect | 招商转化链 | 招商 |
| maintenance-effect | 维护流程影响 | 运营 |

**这已经是 Ontology Event Model 的雏形。**

**缺什么**：每个核心实体的**完整状态机**。

现在你的状态是散落的：


铺位状态：空置、已租、锁定、已退出（business-ontology.yaml 场景描述）
合同状态：草稿、已签、生效、到期、终止（散落在代码 if-else 里）
账单状态：待出账、已出账、已收款、已核销（散落在字段枚举里）


In [ ]:
但 AI 需要的是：


Lease Lifecycle:
  Draft → Signed → Active → Expiring → Expired → Terminated
                                                    ↓
                                              [may re-enter if renewed]

每个迁移点：
  Signed → Active:
    event: lease.activated
    effects:
      - occupancy-effect → Space: status = occupied
      - financial-effect → BillingPlan: generate receivables


In [ ]:
### ✅ 维度 5：Rule（为什么这样变化？）

**你有的**：CRE BCM 14 个域文件里的业务规则（部分显式、部分隐式）。

**缺什么**：规则的**显式声明**，让 AI 可以推理而不是猜。


现在的规则在哪？
  ❌ 在代码 if-else 里：if (lease.status == 'expired' && space.inspection != 'done') { ... }
  ❌ 在业务人员的脑子里
  ✅ 少数在 BCM 域文件里（但格式不统一）

Ontology 需要的格式：
  Rule: "存在未完成退租流程的 Space 不可出租"
  Condition: Space.Lease.status ∈ {Terminating, Terminated} AND Space.inspection != Completed
  Effect: Space.available_for_lease = false


In [ ]:
> **规则藏在代码里，AI 看不见。规则声明在 Ontology 里，AI 才能推理。**

### ✅ 维度 6：Capability（可以做什么？）

**你有的**：business-ontology.yaml 的 capabilities 列表 + CRE BCM capability 行 + capability-traceability-matrix.md。

**缺什么**：Capability 与 Skill（Agent 执行单元）的**映射**。


现在：Capability（asset-resource-rent-control-ui）→ MI 代码模块
缺少：Capability → Agent Skill 映射
      "铺位建档" 这个能力，哪个 Agent 可以执行？需要什么权限？前置条件是什么？


In [ ]:
---

## 三、一张表总结：Gap Analysis

| Ontology 维度 | 已有资产 | 成熟度 | 关键缺口 |
|---|---|---|---|
| Entity | terms 列表 + Object Ownership Matrix | ★★★☆☆ | 缺业务定义（只有术语名，没有"是什么"） |
| Identity | 表 ID + 编码规则 | ★★☆☆☆ | 缺业务身份层级（项目→楼宇→楼层→铺位的 Ontology 路径） |
| Relationship | ADR-006 四类关系 | ★★★★☆ | 缺语义动词命名（occupies / signs / governs） |
| State / Event | effect-registry 5 类 + 散落的状态字段 | ★★★☆☆ | 缺完整状态机 + Event 声明 |
| Rule | BCM 域文件（部分显式） | ★★☆☆☆ | 规则未显式化，AI 不可推理 |
| Capability | capabilities 列表 + BCM + traceability | ★★★☆☆ | 缺 Skill 映射（Capability → Agent 执行单元） |
| Policy | 审批流代码 | ★☆☆☆☆ | 缺 AI 执行约束声明（几乎空白） |

---

## 四、今天改变什么设计判断


以前：business-ontology.yaml 是一份业务术语表和功能清单
现在：business-ontology.yaml 是一个初级 Ontology，
      骨架已对（模块→能力→场景），但需要补六层语义

以前：effect-registry 是一个技术配置文件
现在：effect-registry 是 Ontology Event Model 的注册中心，
      5 类 effect 已经覆盖了状态传播/占用/财务/招商/维护

以前：ADR-006 四类关系是架构文档
现在：ADR-006 四类关系是 Ontology Relationship Model 的治理框架
```

---


## 五、练习（5 分钟）

打开你的 `business-ontology.yaml`，找到"铺位管理"这个子功能。

1. 用 Ontology 六维度，逐项检查"铺位"这个 Entity：
   - 它的业务定义是什么？（Entity）
   - 它的业务身份是什么？（Identity）
   - 它和楼宇、楼层、合同的关系分别叫什么？（Relationship）
   - 它有哪些状态？状态之间怎么迁移？（State/Event）
   - "铺位不可重复出租"——这条规则写在哪？（Rule）
   - "铺位建档"这个 Capability，对应哪个 Agent Skill？（Capability）

2. 把答案写在笔记本上。**不需要完美——写下来本身就是 Ontology 建模的第一步。**

---


## 六、与明天 Day 6 的衔接

明天我们看 **Ontology vs Knowledge Model vs RAG**：

- Ontology 描述"企业有什么"（结构化语义）
- Knowledge Model 描述"什么知识需要被检索"（非结构化知识）
- RAG 描述"如何检索"（技术机制）

三者在 LangChat + MI 里各自什么角色？它们如何配合让 AI 既"懂结构"又"能找资料"？
